# 03 - Graph Neural Network (GCN)
## Predicting Human Intestinal Absorption (HIA)
**Student:** Mohammad Karamath Fardeen | ID: 25251265  
**Supervisor:** Kolawole Adebayo | Maynooth University | 2025-2026

Model: Graph Convolutional Network (GCN) using PyTorch Geometric

In [ ]:
import os, numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from rdkit import Chem
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, matthews_corrcoef

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Molecular Graph Construction

In [ ]:
ATOM_LIST = ['C','N','O','S','F','Cl','Br','I','P','B','Si','H','Other']

def atom_features(atom):
    symbol = atom.GetSymbol()
    enc = [1 if symbol == s else 0 for s in ATOM_LIST[:-1]]
    enc.append(1 if symbol not in ATOM_LIST[:-1] else 0)
    return enc + [atom.GetDegree(), atom.GetFormalCharge(),
                  int(atom.GetHybridization()), int(atom.GetIsAromatic()),
                  atom.GetTotalNumHs()]

def bond_features(bond):
    bt = bond.GetBondType()
    return [int(bt == Chem.rdchem.BondType.SINGLE),
            int(bt == Chem.rdchem.BondType.DOUBLE),
            int(bt == Chem.rdchem.BondType.TRIPLE),
            int(bt == Chem.rdchem.BondType.AROMATIC),
            int(bond.GetIsConjugated()), int(bond.IsInRing())]

def smiles_to_graph(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edge_indices, edge_feats = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        feat = bond_features(bond)
        edge_indices += [[i,j],[j,i]]
        edge_feats   += [feat, feat]
    if not edge_indices:
        edge_index = torch.zeros((2,0), dtype=torch.long)
        edge_attr  = torch.zeros((0,6), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(edge_feats, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.float))

class HIAGraphDataset(InMemoryDataset):
    def __init__(self, csv_path):
        super().__init__(None)
        df = pd.read_csv(csv_path)
        data_list = [g for g in [smiles_to_graph(r['Drug'], int(r['Y']))
                                  for _, r in df.iterrows()] if g is not None]
        print(f'Loaded {len(data_list)} graphs from {csv_path}')
        self.data, self.slices = self.collate(data_list)

print('Graph construction functions defined!')

## 2. GCN Model Architecture

In [ ]:
class GCN(nn.Module):
    def __init__(self, num_node_features, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.conv1 = GCNConv(num_node_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.dropout = dropout
        self.fc1 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc2 = nn.Linear(hidden_dim // 2, 1)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.fc2(x).squeeze(-1)

print('GCN model defined!')

## 3. Train GCN

In [ ]:
def evaluate(model, loader):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            probs = torch.sigmoid(model(batch.x, batch.edge_index, batch.batch))
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend((probs > 0.5).float().cpu().numpy())
            all_labels.extend(batch.y.cpu().numpy())
    return {
        'roc_auc':  round(roc_auc_score(all_labels, all_probs), 4),
        'f1':       round(f1_score(all_labels, all_preds), 4),
        'accuracy': round(accuracy_score(all_labels, all_preds), 4),
        'mcc':      round(matthews_corrcoef(all_labels, all_preds), 4),
    }

# Load datasets
train_ds = HIAGraphDataset('data/raw/hia_train.csv')
valid_ds = HIAGraphDataset('data/raw/hia_valid.csv')
test_ds  = HIAGraphDataset('data/raw/hia_test.csv')

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

num_node_features = train_ds[0].x.shape[1]
model = GCN(num_node_features).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# Class weighting
labels = [int(d.y.item()) for d in train_ds]
n_pos, n_neg = sum(labels), len(labels) - sum(labels)
pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

best_val_auc, best_state, patience_counter = 0.0, None, 0
PATIENCE = 15

print('Training GCN...')
for epoch in range(1, 101):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(batch.x, batch.edge_index, batch.batch), batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs

    val_metrics = evaluate(model, valid_loader)
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d} | Loss: {total_loss/len(train_ds):.4f} | Val AUC: {val_metrics["roc_auc"]:.4f}')

    if val_metrics['roc_auc'] > best_val_auc:
        best_val_auc = val_metrics['roc_auc']
        best_state = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

model.load_state_dict(best_state)
os.makedirs('results/metrics', exist_ok=True)
torch.save(model.state_dict(), 'results/metrics/gcn_model.pt')

test_metrics = evaluate(model, test_loader)
print(f'\nGCN Test Results:')
print(f'AUC: {test_metrics["roc_auc"]} | F1: {test_metrics["f1"]} | MCC: {test_metrics["mcc"]}')